# LH2 v12 anchor fit on Colab\n\nThis runs the CPU-heavy Lighthouse anchor fit on Colab. Free Colab usually has limited CPU cores, so this uses all available CPU workers but keeps BLAS/OpenMP internal threads at 1 to avoid oversubscription.

In [ ]:
import os, multiprocessing\n\nfor name in (\n    'OMP_NUM_THREADS',\n    'OPENBLAS_NUM_THREADS',\n    'MKL_NUM_THREADS',\n    'VECLIB_MAXIMUM_THREADS',\n    'NUMEXPR_NUM_THREADS',\n):\n    os.environ[name] = '1'\n\nworkers = max(1, multiprocessing.cpu_count())\nprint('CPU workers:', workers)\n!python -V

## Get the project\n\nIf the GitHub repo is accessible from Colab, use the clone cell. Otherwise upload a zip containing `positioningv12/` using the upload cell below.

In [ ]:
# Option A: clone the repo/branch\n# Replace the URL if needed. If the repo is private, use Colab's GitHub auth or upload a zip instead.\n!rm -rf /content/lh2_positioning\n!git clone --branch lbees2 https://github.com/LBOrganizationWorkArea/lh2_on_rp2040.git /content/lh2_positioning\n%cd /content/lh2_positioning/positioningv12\n!ls

In [ ]:
# Option B: upload a zip instead of git clone. Run this cell only if Option A is not usable.\n# The zip should contain a positioningv12 folder.\n# from google.colab import files\n# uploaded = files.upload()\n# import zipfile, pathlib\n# zip_name = next(iter(uploaded))\n# !rm -rf /content/lh2_positioning\n# pathlib.Path('/content/lh2_positioning').mkdir(exist_ok=True)\n# with zipfile.ZipFile(zip_name) as z:\n#     z.extractall('/content/lh2_positioning')\n# %cd /content/lh2_positioning/positioningv12

## Install dependencies

In [ ]:
!pip -q install numpy scipy pyserial opencv-python-headless\n!python - <<'PY'\nimport numpy, scipy\nprint('numpy', numpy.__version__)\nprint('scipy', scipy.__version__)\nPY

## Run the anchor fit\n\nThis uses your 7 vertical anchor points and the current PnP relative block file. If you have a manual measured BS10 translation, replace the `--relative-geometry ...` part with `--bs10-translation X,Y,Z`.

In [ ]:
import multiprocessing, os\nworkers = max(1, multiprocessing.cpu_count())\n\ncmd = f'''python tools/23_anchor_lighthouse_frame_to_room.py \\\n  --relative-geometry config/lighthouse_relative_pnp_oldwave_cluster_d14.json \\\n  --anchor-poses config/vertical_pose_variants/vertical_x_up_face_minus_y.json \\\n  --max-anchor-spread-deg 0.5 \\\n  --starts 4 \\\n  --workers {workers} \\\n  --max-nfev 700 \\\n  --convention-search all \\\n  --output config/lighthouse_anchor_from_measured_block_colab.json'''\n\nprint(cmd)\n!{cmd}

## Save/download the result

In [ ]:
from google.colab import files\nfiles.download('config/lighthouse_anchor_from_measured_block_colab.json')